Когда мы вставляем данные в базу, иногда нам нужно получить идентификатор вставленной записи и потом использовать его в коде. Например, когда мы создаем какую-то сущность и хотим потом ее использовать:

```python
user = User()
# Сохраняем пользователя в базу данных
# После этого становится доступен id
id = user.get_id()
# Его можно использовать для формирования ссылок или вставки связанных записей
```

`PostgreSQL` поддерживает синтаксис `RETURNING`, который возвращает указанные поля вставленной строки:

```python
with conn.cursor() as cur:
    cur.execute(
        "INSERT INTO users (name, email) VALUES (%s, %s) RETURNING id;",
        (user.name, user.email),
    )
    user.id = cur.fetchone()[0]
conn.close()
```

`Что делает RETURNING`

В `PostgreSQL` конструкция `RETURNING` позволяет вернуть данные из строки, которую ты только что вставил (или обновил/удалил). В случае `INSERT` это чаще всего `id`, но можно вернуть и другие поля:

```sql
INSERT INTO users (name, email) VALUES ('Bob', 'bob@mail.com')
RETURNING id;
```

или даже:

```sql
INSERT INTO users (name, email) VALUES ('Bob', 'bob@mail.com')
RETURNING id, name, created_at;
```

Это удобно тем, что:

- не нужно делать отдельный `SELECT` по каким‑то уникальным полям;
- нет риска получить не ту строку (например, если у двух пользователей совпадут имя и почта);
- это атомарно: вставка и возврат происходят в рамках одной операции.